## Training Strategy

We will use a **Multinomial Naive Bayes classifier** with a **Bag-of-Words representation** to classify emails as either spam or ham.

### How Naive Bayes Works

Naive Bayes calculates the **prior probability** of each class and the **likelihood** of observing the words in a document given that class. These values are then combined to estimate the posterior probability of each class.

The classifier returns the argmax between each posterior probability.

### Equations

#### 1. Prior Probability

The prior probability represents how frequently a class occurs within the training corpus.

$$P(c) = \frac{D_c}{D}$$

Where:

* ${D_c}$ is the number of documents belonging to class $c$.
* ${D}$ is the total number of documents in the corpus.

#### 2. Word Likelihood

The likelihood represents the probability of observing a particular word given a class.

$$P(w_i \mid c) = \frac{N_{w_i,c} + \alpha}{N_c + \alpha V}$$

Where:

* $N_{w_i,c}$ is the number of times word $w_i$ appears in class $c$.
* $N_c$ is the total number of word occurrences in class $c$, including repetitions.
* $V$ is the total vocabulary size, excluding repetitions.
* $\alpha$ is the smoothing parameter used to prevent zero likelihoods.

Laplace smoothing ensures that a word that does not occur in a particular class does not cause the entire posterior probability to become zero.

#### 3. Posterior Probability

Given a document represented by its words $X_1, X_2, \ldots, X_n$, Naive Bayes estimates the probability of each class using:

$$P(c \mid X) \propto P(c)\prod_{i=1}^{n}P(X_i \mid c)$$

The denominator from Bayes' theorem is omitted because it is identical for every candidate class and therefore does not affect which class has the highest probability.

The predicted class is therefore:

$$\hat{c} = \underset{c}{\operatorname{argmax}}\left[P(c)\prod_{i=1}^{n}P(X_i \mid c)\right]$$

### Arithmetic Underflow

Directly multiplying the likelihoods can result in extremely small numbers. As the number of words in a document increases, the product of many probabilities between 0 and 1 can become smaller than the range representable by standard floating-point numbers.

This results in **arithmetic underflow**, where the calculated probability underflows to (0.0).

To avoid this, we transform the calculation from probability space into **log space**.

Using the logarithmic identity:

$$\log(A \times B) = \log(A) + \log(B)$$

we can transform the posterior calculation:

$$\log\left(P(c)\prod_{i=1}^{n}P(X_i \mid c)\right) = \log P(c) + \sum_{i=1}^{n}\log P(X_i \mid c)$$

Therefore, the log-space scoring function becomes:

$$\log P(c \mid X) \propto \log P(c) + \sum_{i=1}^{n}\log P(X_i \mid c)$$

The class with the highest log-probability is selected as the prediction.

\log P(c \mid X) \propto \log P(c) + \sum_{i=1}^{n}\log P(X_i \mid c)$$

The class with the highest log-probability is selected as the prediction.

In [2]:
# Imports 
import json
import re
from collections import Counter

import numpy as np
import pandas as pd

In [3]:
# Load data and drop null values 
df = pd.read_csv("../data/raw/training_data.csv")
df = df.dropna(subset=["text"])

## Cleaning strategy

Two ablation experiments are run to observe how preprocessing choices 
affect precision, recall, F1, and accuracy.

### Experiment 1: nested ablation (v1–v4)
The same cleaning strategy from the EDA notebook is used as a baseline 
(v1). Four configurations are trained and evaluated, each removing one 
additional filtering layer on top of the previous version, ranging from 
full filtering (v1) down to minimal filtering (v4).

| Version | Stopwords removed? | NOISE_WORDS removed? | escapenumber/escapelong stripped? | Punctuation stripped? |
|---------|--------------------|-----------------------|------------------------------------|------------------------|
| v1      | Yes                | Yes                   | Yes                                 | Yes                    |
| v2      | Yes                | No                     | Yes                                 | Yes                    |
| v3      | No                 | No                     | Yes                                 | Yes                    |
| v4      | No                 | No                     | No                                  | Yes                    |

This nested design shows the effect of removing filtering layers *in 
this particular combined order*, but cannot isolate which individual 
factor is responsible for any observed trend, e.g. whether stopword 
removal alone behaves differently than stopword removal combined with 
noise-word removal.

### Experiment 2: factorial ablation (v1–v8)
To isolate each factor's individual and combined effect, all 2³ = 8 
combinations of the three binary filtering choices (stopword removal, 
noise-word removal, placeholder-token stripping) are tested 
independently, rather than cumulatively.

| Config | Placeholders stripped? | Stopwords removed? | NOISE_WORDS removed? |
|--------|--------------------------|----------------------|------------------------|
| v1     | Yes                      | Yes                  | Yes                    |
| v2     | Yes                      | Yes                  | No                     |
| v3     | Yes                      | No                   | Yes                    |
| v4     | Yes                      | No                   | No                     |
| v5     | No                       | Yes                  | Yes                    |
| v6     | No                       | Yes                  | No                     |
| v7     | No                       | No                   | Yes                    |
| v8     | No                       | No                   | No                     |

Configs v1, v2, v4, and v8 in this experiment correspond exactly to v1, 
v2, v3, and v4 from Experiment 1, respectively, allowing the two 
experiments' results to be cross-checked against each other.

Results for both experiments are presented in the evaluation notebook.


In [25]:
# clean and tokanize
pattern = re.compile(r'[^a-zA-Z\s]+')
noise_pattern = re.compile(r'escapenumber|escapelong', re.IGNORECASE)
stopwords = {'y', 'did', 'i', 'again', "hadn't", 'my', 'over', 'too', 'here', "that'll", 'couldn', 'than', "they've", 'same', "she's", 'they', 'doing', 'if', 'down', "don't", 'out', 'no', 'her', 're', 'such', 't', "i've", 'only', 'was', 'shouldn', "they'd", 'won', 'more', 'needn', 'do', "shouldn't", "wasn't", "aren't", 'aren', 'how', 'shan', 'doesn', 'few', 'll', 'myself', "couldn't", "he'd", 'other', 'wasn', 'his', 'on', 'these', 'both', "didn't", 'you', "you've", 'their', 'what', "you'd", 'being', 'by', 'been', 's', 'ain', 'why', 'where', 'until', 'as', 'off', 'after', 'is', 'be', 'below', 'hasn', 'yours', 'we', 'has', 'own', 've', 'haven', 'whom', 'wouldn', 'hers', "we'd", 'between', 'o', "he's", 'have', 'herself', 'does', 'now', "shan't", 'don', 'in', 'so', 'from', 'a', 'under', 'further', 'those', 'me', 'most', "weren't", 'against', 'its', 'she', 'at', "you'll", 'yourself', "i'll", 'm', 'once', 'mustn', 'while', 'should', 'ours', 'didn', 'then', 'when', 'that', 'were', "it'd", 'which', 'above', 'all', "doesn't", "mustn't", "she'd", "it's", 'before', 'of', 'and', 'weren', "she'll", 'our', 'will', 'isn', "i'm", 'had', 'to', 'the', 'through', 'with', 'there', 'during', 'or', "haven't", 'himself', 'it', 'theirs', "wouldn't", 'each', 'not', 'just', 'this', "we'll", "he'll", "needn't", "we've", 'ourselves', 'about', "isn't", 'your', 'nor', 'because', 'can', 'he', 'am', "i'd", "should've", 'any', 'd', 'some', 'having', 'ma', 'itself', 'into', "we're", "hasn't", "they'll", 'who', 'are', 'but', 'themselves', "mightn't", "it'll", 'very', 'for', 'hadn', "you're", 'him', 'them', 'an', "they're", 'mightn', "won't", 'yourselves', 'up'}
NOISE_WORDS = {"subject", "pm", "org", "com", "http", "www"}

def clean(email: str, strip_placeholders: bool = True, remove_stopwords: bool = True, remove_noise_words: bool = True) -> list[str]:
    if strip_placeholders:
        email = noise_pattern.sub('', email)
    
    words = email.lower().split()
    words = [pattern.sub('', w) for w in words]
    
    words = [
        w for w in words
        if w and len(w) > 1
        and (not remove_stopwords or w not in stopwords)
        and (not remove_noise_words or w not in NOISE_WORDS)
    ]
    return words



In [14]:
def clean_params(version: int = 1) -> tuple[bool, bool, bool]:
    placeholders, stopw, noisew = True, True, True

    match version:
        case 1:
            pass
        case 2:
            noisew = False
        case 3:
            stopw = False
        case 4:
            stopw, noisew = False, False
        case 5:
            placeholders = False
        case 6:
            placeholders, noisew = False, False
        case 7:
            placeholders, stopw = False, False
        case 8:
            placeholders, stopw, noisew = False, False, False
        case _:
            raise ValueError(f"Unknown version: {version}")

    return placeholders, stopw, noisew
        

In [16]:
spam_column = df[df["label"] == "Spam"]
spam_text_column = spam_column["text"]

ham_column = df[df["label"] == "Ham"]
ham_text_column = ham_column["text"]

In [18]:
# build vocabulary 
def vocab(version: int = 1):
    placeholders, stopw, noisew = clean_params(version)

    spam_vocab = []
    ham_vocab = []

    for ham_email in ham_text_column:
        ham_vocab.extend(clean(ham_email, placeholders, stopw, noisew))

    for spam_email in spam_text_column:
        spam_vocab.extend(clean(spam_email, placeholders, stopw, noisew))

    global_vocab = set(ham_vocab) | set(spam_vocab)

    return global_vocab, spam_vocab, ham_vocab

In [22]:
# Compute priors 
corpus = len(df)
prior_spam = np.log(len(spam_column) / corpus) # log P(spam)
prior_ham = np.log(len(ham_column) / corpus) # log P(ham)


In [23]:
def train(version: int):
    global_vocab, spam_vocab, ham_vocab = vocab(version)

    V = len(global_vocab)
    indexed_vocab = {word: i for i, word in enumerate(global_vocab)}
    indexed_vocab

    alpha = 1
    spam_likelihoods = np.zeros(V)
    ham_likelihoods = np.zeros(V)

    Ns = len(spam_vocab) # total number of words in spam, repititions 
    Nh = len(ham_vocab) # total number of words in ham, repititions 

    spam_word_count = Counter(spam_vocab)
    ham_word_count = Counter(ham_vocab)

    for word, index in indexed_vocab.items():
        Nws = spam_word_count.get(word, 0)
        Nwh = ham_word_count.get(word, 0)

        spam_likelihoods[index] = np.log((Nws + alpha) / (Ns + V * alpha))
        ham_likelihoods[index] = np.log((Nwh + alpha) / (Nh + V * alpha))

    log_likelihoods = np.vstack([spam_likelihoods, ham_likelihoods])

    model = {
    "vocab": indexed_vocab,
    "log_priors": [float(prior_spam), float(prior_ham)],
    "log_likelihoods": log_likelihoods.tolist(),
    }

    path = f"../models/modelv{version}.json"
    with open(path, "w") as f:
        json.dump(model, f, indent=2)

    return path, V

In [24]:
# save model to model.json
results = []
for v in range(1, 9):
    path, vocab_size = train(v)
    results.append({"version": v, "path": path, "vocab size": vocab_size})
    print(f"Trained v{v} -> {path} (vocab size: {vocab_size})")

Trained v1 -> ../models/modelv1.json (vocab size: 364103)
Trained v2 -> ../models/modelv2.json (vocab size: 364109)
Trained v3 -> ../models/modelv3.json (vocab size: 364248)
Trained v4 -> ../models/modelv4.json (vocab size: 364254)
Trained v5 -> ../models/modelv5.json (vocab size: 376162)
Trained v6 -> ../models/modelv6.json (vocab size: 376168)
Trained v7 -> ../models/modelv7.json (vocab size: 376307)
Trained v8 -> ../models/modelv8.json (vocab size: 376313)


In [27]:
# Prediction
def load_model(path="../models/modelv4.json"):
    with open(path, "r") as f:
        model = json.load(f)
    vocab = model["vocab"]
    log_prior_s, log_prior_h = np.array(model["log_priors"])
    log_likelihoods = np.array(model["log_likelihoods"])
    return vocab, log_prior_s, log_prior_h, log_likelihoods

def predict(email, vocab, log_prior_s, log_prior_h, log_likelihoods, clean_fn):
    cleaned = clean_fn(email)
    
    log_post_spam = log_prior_s
    log_post_ham = log_prior_h
    
    for word in cleaned:
        index = vocab.get(word)
        if index is None:
            continue
        log_post_spam += log_likelihoods[0][index]
        log_post_ham += log_likelihoods[1][index]

    return "spam" if log_post_spam > log_post_ham else "ham"

In [29]:
# sanity checks 
sample_df = pd.read_csv("../data/sample/sample_dataset.csv")
correct = 0
total = len(sample_df)

vocab, log_prior_s, log_prior_h, log_likelihoods = load_model()

In [30]:
for version in range(1, 9):
    path = f"../models/modelv{version}.json"
    vocab, log_prior_s, log_prior_h, log_likelihoods = load_model(path)
    clean_fn = lambda email, v=version: clean(email, *clean_params(v))
    
    correct = 0
    total = len(sample_df)
    
    for label, text in zip(sample_df["label"], sample_df["text"]):
        prediction = predict(text, vocab, log_prior_s, log_prior_h, log_likelihoods, clean_fn)
        if prediction == label.lower():
            correct += 1
    
    accuracy = (correct / total) * 100
    print(f"v{version}: {accuracy:.2f}% ({correct}/{total})")


v1: 96.99% (548/565)
v2: 96.99% (548/565)
v3: 96.81% (547/565)
v4: 96.99% (548/565)
v5: 96.81% (547/565)
v6: 96.81% (547/565)
v7: 96.64% (546/565)
v8: 96.64% (546/565)
